In [7]:
import psycopg
from typing import List, Dict, Tuple
import json
import os


def get_connection():
    """Get a single database connection."""
    conn = psycopg.connect(
        host='localhost',
        port=5432,
        user='postgres',
        password='12345678',
        dbname='clinc',
        client_encoding='utf8'
    )
    print("Connected to database")
    return conn

In [8]:
conn = get_connection()

Connected to database


In [ ]:
from pydantic import BaseModel, Field, EmailStr
from pydantic import ConfigDict
from datetime import date
from enum import Enum
from typing import Annotated
class PatientCreate(BaseModel):
    """
    Payload for POST /patients  (assistant or doctor).
    first_name and last_name are required — all other fields are optional
    to accommodate incomplete registration at walk-in.
    """

    model_config = ConfigDict(from_attributes=True)
    id : str = Field(...)
    first_name: str = Field(..., max_length=100)
    last_name:  str = Field(..., max_length=100)

    date_of_birth: date | None = None
    gender:        str | None = None

    national_id: str | None = Field(
        default=None,
        max_length=50,
        description="National identity number. Must be unique if provided.",
    )
    phone: str | None = Field(default=None, max_length=30)


    email: Annotated[str, EmailStr] | None = Field(default=None,
                                                   escription="Optional contact email. Not used for system authentication.",
                                                   )
    address: str | None = None

/tmp/ipykernel_67944/3033134936.py:29: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'escription'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  email: Annotated[str, EmailStr] | None = Field(default=None,


In [23]:
def create_new_doctor(first_name: str, last_name: str, date_of_birth: date | None,
                      gender: str | None, national_id: str | None,
                      phone: str | None, email: str | None, address: str | None, id: int) -> int:
    try:
        with conn.cursor() as cur:
            cur.execute(
                """
                INSERT INTO patients (id, first_name, last_name, date_of_birth, gender, national_id, phone, email, address)
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
                RETURNING id;
                """,
                (
                    id,
                    first_name,
                    last_name,
                    date_of_birth,
                    gender,
                    national_id,
                    phone,
                    email,
                    address
                )
            )
            new_id = cur.fetchone()[0]
        conn.commit()
        return new_id
    except Exception:
        conn.rollback()
        raise

In [24]:
class Gender(Enum):
    MALE = "male"
    

In [25]:
create_new_doctor(
    id=1,
    first_name="John",
    last_name="Doe",
    date_of_birth=date(1990, 1, 1),
    gender=Gender.MALE,
    national_id="123456789",
    phone="555-1234",
    email="jjjs",
    address="123 Main St"
)

1